# *DATASET IMPORT FROM THE KAGGLE AND LOAD IT IN DATAFRAME*

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

import os
print(os.listdir(path))

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews
['IMDB Dataset.csv']


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv(path+'/IMDB Dataset.csv')

In [ ]:
df['review'][0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

# *DATA PREPROCESSING*

In [ ]:
import re

In [ ]:
# To remove the html tags
def remove_html_tags(text):
  patObj = re.compile('<.*?>')
  return(patObj.sub('', text).lower())

In [ ]:
print(remove_html_tags("r far away.<br /><br />I would say the"))

r far away.i would say the


In [ ]:
# To remove punctuation
import string
punc = string.punctuation
def remove_punc(text):
  for char in punc:
    text = text.replace(char, '')
  return text

In [ ]:
print(remove_punc("r far away.i would say the"))

r far awayi would say the


In [ ]:
# to remove the chatwords

chat_words = {
    "LOL": "Laughing Out Loud",
    "BRB": "Be Right Back",
    "BTW": "By The Way",
    "IDK": "I Don’t Know",
    "IMO": "In My Opinion",
    "IMHO": "In My Humble Opinion",
    "TTYL": "Talk To You Later",
    "OMG": "Oh My God",
    "FYI": "For Your Information",
    "ASAP": "As Soon As Possible",
    "BFF": "Best Friends Forever",
    "DM": "Direct Message",
    "JK": "Just Kidding",
    "NP": "No Problem",
    "ROFL": "Rolling On the Floor Laughing",
    "SMH": "Shaking My Head",
    "TBF": "To Be Fair",
    "ICYMI": "In Case You Missed It",
    "AFK": "Away From Keyboard",
    "YOLO": "You Only Live Once"
}
def remove_ChatWords(text):
  new_text = []
  patternText = re.compile('\w+|[^\w\s]')
     # to get (words) and (not words or not spaces i.e. special symbols) as separate tokens
  newText = patternText.findall(text)
  for w in newText:
  # for w in text.split():
    if w.upper() in chat_words:
      new_text.append(chat_words[w.upper()])
    else:
      new_text.append(w)
  return " ".join(new_text).lower()

<>:27: SyntaxWarning: invalid escape sequence '\w'
<>:27: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipython-input-324745237.py:27: SyntaxWarning: invalid escape sequence '\w'
  patternText = re.compile('\w+|[^\w\s]')


In [ ]:
print(remove_ChatWords("lol r far away.i would say the"))

laughing out loud r far away . i would say the


In [ ]:
# Stop word removal

import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords

stopWords= stopwords.words('english')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
def remove_StopWords(text):
  ## This commented section was supposed to be not needed
  ## since we have the punctuation already removed
  ## but remember it will remove any pronouns and such which may be
  ## important for the pos tagging
  patObj = re.compile('[\w\']+|[^\w\s]')
  tokens = patObj.findall(text)
  newText = []
  for words in tokens:
  # for words in text.split():
    if words not in stopWords:
      newText.append(words)
  # print(newText)
  return  " ".join(newText)
  # return newText

<>:6: SyntaxWarning: invalid escape sequence '\w'
<>:6: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipython-input-4052053676.py:6: SyntaxWarning: invalid escape sequence '\w'
  patObj = re.compile('[\w\']+|[^\w\s]')


In [ ]:
print(remove_StopWords("lol r far away.i would say the"))

lol r far away . would say


In [ ]:
# Lemmatization

nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
  # Note needs to be in lower case to work properly
from nltk.stem import WordNetLemmatizer
wnlObj = WordNetLemmatizer()

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


In [ ]:
import spacy
dictionary = spacy.load('en_core_web_sm')

In [ ]:
def tokenizeWordLevel(text):
  tokens = dictionary(text)
  out = []
  for token in tokens:
    out.append(str(token))
  return out

# LEMMATIZATION
  # Note needs to be in lower case to work properly
from nltk.stem import WordNetLemmatizer
wnlObj = WordNetLemmatizer()
def posLabelForLemmatize(pos):
  if pos.startswith('J'): # adjective
    return 'a'
  elif pos.startswith('V'): # verb
    return 'v'
  elif pos.startswith('N'): # noun
    return 'n'
  elif pos.startswith('R'): # adverb
    return 'r'
  else:
    return 'n'
def lemmatizer(text):
  tokens = tokenizeWordLevel(text)
  # print(tokens)
  token_pos_list = nltk.pos_tag(tokens)
  output = []
  for token, pos in token_pos_list:
    # print(token, pos)
    output.append(wnlObj.lemmatize(token, pos= posLabelForLemmatize(pos)))
  return " ".join(output)

In [ ]:
print(nltk.pos_tag(['running']))
print(nltk.pos_tag(['dancing']))

print(lemmatizer("lol r far away.i would say the running dancing was unnecessary did not i"))
# not 100% accurate

[('running', 'VBG')]
[('dancing', 'VBG')]
lol r far away.i would say the run dancing be unnecessary do not i


### applying the text preprocessing

In [ ]:
df.head()

,review,sentiment,reviewNoHtml,reviewNoPunc,reviewNoChatWords,reviewNoStopWords
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,one reviewers mentioned watching 1 oz episode ...
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production. the filming tec...,a wonderful little production the filming tech...,a wonderful little production the filming tech...,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,negative,basically there's a family where a little boy ...,basically theres a family where a little boy j...,basically theres a family where a little boy j...,basically theres family little boy jake thinks...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"petter mattei's ""love in the time of money"" is...",petter matteis love in the time of money is a ...,petter matteis love in the time of money is a ...,petter matteis love time money visually stunni...


In [ ]:
df['reviewNoHtml'] = df['review'].apply(remove_html_tags)

In [ ]:
df['reviewNoPunc'] = df['reviewNoHtml'].apply(remove_punc)

In [ ]:
df['reviewNoChatWords'] = df['reviewNoPunc'].apply(remove_ChatWords)

In [ ]:
df['reviewNoStopWords'] = df['reviewNoChatWords'].apply(remove_StopWords)

In [ ]:
df

,review,sentiment,reviewNoHtml,reviewNoPunc,reviewNoChatWords,reviewNoStopWords
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,one reviewers mentioned watching 1 oz episode ...
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production. the filming tec...,a wonderful little production the filming tech...,a wonderful little production the filming tech...,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,negative,basically there's a family where a little boy ...,basically theres a family where a little boy j...,basically theres a family where a little boy j...,basically theres family little boy jake thinks...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"petter mattei's ""love in the time of money"" is...",petter matteis love in the time of money is a ...,petter matteis love in the time of money is a ...,petter matteis love time money visually stunni...
...,...,...,...,...,...,...
49995,I thought this movie did a down right good job...,positive,i thought this movie did a down right good job...,i thought this movie did a down right good job...,i thought this movie did a down right good job...,thought movie right good job wasnt creative or...
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative,"bad plot, bad dialogue, bad acting, idiotic di...",bad plot bad dialogue bad acting idiotic direc...,bad plot bad dialogue bad acting idiotic direc...,bad plot bad dialogue bad acting idiotic direc...
49997,I am a Catholic taught in parochial elementary...,negative,i am a catholic taught in parochial elementary...,i am a catholic taught in parochial elementary...,i am a catholic taught in parochial elementary...,catholic taught parochial elementary schools n...
49998,I'm going to have to disagree with the previou...,negative,i'm going to have to disagree with the previou...,im going to have to disagree with the previous...,im going to have to disagree with the previous...,im going disagree previous comment side maltin...


In [ ]:
dfNew = pd.DataFrame({})
dfNew['textSample']= df['reviewNoStopWords'].iloc[:1000]
import time
start = time.time()
dfNew['lemmatized'] = dfNew['textSample'].apply(lemmatizer)
end = time.time()
print(f"{(end-start)*50/60}+min")

22.430618405342102+min


In [ ]:
df

,review,sentiment,reviewNoHtml,reviewNoPunc,reviewNoChatWords,reviewNoStopWords
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,one reviewers mentioned watching 1 oz episode ...
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production. the filming tec...,a wonderful little production the filming tech...,a wonderful little production the filming tech...,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,negative,basically there's a family where a little boy ...,basically theres a family where a little boy j...,basically theres a family where a little boy j...,basically theres family little boy jake thinks...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"petter mattei's ""love in the time of money"" is...",petter matteis love in the time of money is a ...,petter matteis love in the time of money is a ...,petter matteis love time money visually stunni...
...,...,...,...,...,...,...
49995,I thought this movie did a down right good job...,positive,i thought this movie did a down right good job...,i thought this movie did a down right good job...,i thought this movie did a down right good job...,thought movie right good job wasnt creative or...
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative,"bad plot, bad dialogue, bad acting, idiotic di...",bad plot bad dialogue bad acting idiotic direc...,bad plot bad dialogue bad acting idiotic direc...,bad plot bad dialogue bad acting idiotic direc...
49997,I am a Catholic taught in parochial elementary...,negative,i am a catholic taught in parochial elementary...,i am a catholic taught in parochial elementary...,i am a catholic taught in parochial elementary...,catholic taught parochial elementary schools n...
49998,I'm going to have to disagree with the previou...,negative,i'm going to have to disagree with the previou...,im going to have to disagree with the previous...,im going to have to disagree with the previous...,im going disagree previous comment side maltin...


In [ ]:
df['reviewLemmatized'] = df['reviewNoStopWords'].apply(lemmatizer)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
csv_path = "/content/drive/MyDrive/preprocessed csv/data.csv"
df.to_csv(csv_path, index=False)  # index=False avoids writing row numbers

In [ ]:
import pandas as pd

csv_path = "/content/drive/MyDrive/preprocessed csv/data.csv"
df_loaded = pd.read_csv(csv_path)
df_loaded.head() # show first 5 rows


,review,sentiment,reviewNoHtml,reviewNoPunc,reviewNoChatWords,reviewNoStopWords,reviewLemmatized
0,One of the other reviewers has mentioned that ...,positive,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...,one reviewers mentioned watching 1 oz episode ...,one reviewer mention watch 1 oz episode you ll...
1,A wonderful little production. <br /><br />The...,positive,a wonderful little production. the filming tec...,a wonderful little production the filming tech...,a wonderful little production the filming tech...,wonderful little production filming technique ...,wonderful little production film technique una...
2,I thought this was a wonderful way to spend ti...,positive,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...,think wonderful way spend time hot summer week...
3,Basically there's a family where a little boy ...,negative,basically there's a family where a little boy ...,basically theres a family where a little boy j...,basically theres a family where a little boy j...,basically theres family little boy jake thinks...,basically there s family little boy jake think...
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive,"petter mattei's ""love in the time of money"" is...",petter matteis love in the time of money is a ...,petter matteis love in the time of money is a ...,petter matteis love time money visually stunni...,petter matteis love time money visually stunni...
